# Creating Hamiltonians

Hamiltonians are stored in a `Hamiltonian` object. We can either create them from scratch, or use built-in methods to generate common models. For example, to create a Hubbard model in 2D we can use the `hubbard_2d` method:

In [1]:
import warnings
warnings.filterwarnings("ignore")
from trovolut.qm.hamiltonians.models import hubbard_2d

hamiltonian = hubbard_2d(Lx=3, Ly=3, t_hop=1.0, U_onsite=4.0, sparse=True)
for term in hamiltonian.decompose()['terms'][:5]:
    print(term)

[0 - 1fcd61f00]    0.000000 {5}{module_config}: Module numa can not detect resources.
{'coeff': np.float64(-0.5), 'pauli': 'X0 Z1 X2'}
{'coeff': np.float64(-0.5), 'pauli': 'Y0 Z1 Y2'}
{'coeff': np.float64(-0.5), 'pauli': 'X1 Z2 X3'}
{'coeff': np.float64(-0.5), 'pauli': 'Y1 Z2 Y3'}
{'coeff': np.float64(-0.5), 'pauli': 'X2 Z3 X4'}


To create the Hamiltonian of a molecule, we can use the `ElectronicHamiltonian` class, which automatically constructs the second-quantized Hamiltonian based on the molecular geometry and basis set.

In [2]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np  

from trovolut.qm.hamiltonians import ElectronicHamiltonian

First, we use `RDKit` to generate an `.xyz` file for dimethyl disulfide $\mathrm{C_2H_6S_2}$

In [3]:
dmds_smiles = "CSSC"

# RDKit: SMILES -> mol
mol = Chem.MolFromSmiles(dmds_smiles)
mol = Chem.AddHs(mol)  # add hydrogens

# embed 3D coordinates
AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
AllChem.UFFOptimizeMolecule(mol)

# extract XYZ as multiline string in standard XYZ format:
atoms = [atom.GetSymbol() for atom in mol.GetAtoms()]
conf = mol.GetConformer()

xyz_lines = []
xyz_lines.append(str(len(atoms)))
xyz_lines.append("dmds generated by RDKit")
for i, sym in enumerate(atoms):
    pos = conf.GetAtomPosition(i)
    xyz_lines.append(f"{sym} {pos.x:.8f} {pos.y:.8f} {pos.z:.8f}")

dmds_xyz = "\n".join(xyz_lines)

print(dmds_xyz.splitlines()[0:6])  # show header + first few atoms

['10', 'dmds generated by RDKit', 'C -1.71329032 -0.19729145 -0.31167220', 'S -1.02697318 0.54829430 1.19781729', 'S 0.56609306 -0.86673302 1.32841144', 'C 1.75165668 0.22380009 0.48540253']


We then load the geometry to the `ElectronicHamiltonian` class and specify the basis, mapping, and whether to freeze core orbitals. 

In [4]:
H_elec = ElectronicHamiltonian.from_xyz(
    xyz_source=dmds_xyz,
    basis="sto-3g",
    charge=0,
    spin_mult=1,
    use_gpu4pyscf=False,
    freeze_core=True,
)

# basic metadata
meta = H_elec.decompose()
print("n spin orbitals:", meta["n_spin_orbs"])
print("constant nuclear repulsion:", meta["constant"])
print("h1 shape:", np.array(meta["one_body"]).shape)
print("h2 shape:", np.array(meta["two_body"]).shape)

converged SCF energy = -864.654938672564
n spin orbitals: 16
constant nuclear repulsion: 221.521270084568
h1 shape: (16, 16)
h2 shape: (16, 16, 16, 16)


We can then convert the electronic Hamiltonian to a qubit Hamiltonian using the Jordan-Wigner mapping, represented by the `QElectronicHamiltonian` class.

In [5]:
H_qubit = H_elec.to_qubits(mapping="jordan-wigner", sparse=True)

qmeta = H_qubit.decompose()
print("n_qubits:", qmeta["n_qubits"])
print("mapping:", qmeta["mapping"])
print("number of Pauli terms:", len(qmeta["terms"]))
print("first 5 terms:")
for term in qmeta["terms"][:5]:
    print(term)

n_qubits: 16
mapping: jordan-wigner
number of Pauli terms: 4657
first 5 terms:
{'coeff': np.float64(89.88125911910353), 'pauli': 'I'}
{'coeff': np.float64(8.173752515440594), 'pauli': 'Z0'}
{'coeff': np.float64(-2.601594466867398e-07), 'pauli': 'Y0Z1Y2'}
{'coeff': np.float64(-2.601594466867398e-07), 'pauli': 'X0Z1X2'}
{'coeff': np.float64(1.675881825261904e-07), 'pauli': 'Y0Z1Z2Z3Y4'}
